# JMIR multilingual evidence-selection experiment — inference

CPU-first, resumable inference for the frozen 10-case pilot. This notebook runs **two smoke-test cases first** and will not start the full pilot until you set an approval flag.

Design per model: 10 cases × 3 languages × 3 evidence conditions = **90 trials**. Run Qwen and Gemma separately. Results are appended to Drive after every trial.

## 1. Mount Drive and validate the frozen casebook
Expected file: `MyDrive/JMIR_Evidence_Selection/casebooks/pilot_cases_frozen.csv`.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import base64, hashlib, json, os, re, subprocess, time, urllib.request, urllib.error
from datetime import datetime, timezone
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection')
CASEBOOK = ROOT / 'casebooks' / 'pilot_cases_frozen.csv'
HASHFILE = ROOT / 'casebooks' / 'pilot_cases_frozen.csv.sha256'
RESULTS = ROOT / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

assert CASEBOOK.exists(), f'Missing: {CASEBOOK}'
assert HASHFILE.exists(), f'Missing: {HASHFILE}'
expected_hash = HASHFILE.read_text().strip().split()[0]
actual_hash = sha256_file(CASEBOOK)
assert actual_hash == expected_hash, f'Casebook hash mismatch: {actual_hash} != {expected_hash}'
cases = pd.read_csv(CASEBOOK, dtype=str, keep_default_na=False)
assert len(cases) == 10, f'Expected 10 pilot cases, found {len(cases)}'
print('Verified casebook:', len(cases), 'cases')
print('SHA256:', actual_hash)
display(cases[['case_id','dataset','reference_answer','question_en','question_id','question_ms']])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verified casebook: 10 cases
SHA256: a87892688513f68416d29322ab835eab26aa184eaa3906655b6cbc8d95de3981


,case_id,dataset,reference_answer,question_en,question_id,question_ms
0,C001,vqa_rad,yes,is there contrast used in the above image?,Apakah bahan kontras digunakan pada citra di a...,Adakah bahan kontras digunakan dalam imej di a...
1,C002,vqa_rad,no,are the opacities confined to one lung lobe?,Apakah opasitas terbatas pada satu lobus paru?,Adakah kelegapan terhad kepada satu lobus paru...
2,C003,vqa_rad,yes,is anything wrong in the image?,Apakah terdapat kelainan pada citra ini?,Adakah terdapat sebarang keabnormalan dalam im...
3,C004,vqa_rad,no,is there a fracture?,Apakah terdapat fraktur?,Adakah terdapat fraktur?
4,C005,vqa_rad,yes,is this a noncontrast ct?,Apakah ini merupakan CT tanpa kontras?,Adakah ini imbasan CT tanpa kontras?
5,C006,vqa_rad,yes,are there increased vascular markings,Apakah terdapat peningkatan corakan vaskular p...,Adakah terdapat peningkatan corak vaskular pul...
6,C007,vqa_rad,yes,is this an axial image?,Apakah ini merupakan citra aksial?,Adakah ini imej aksial?
7,C008,vqa_rad,no,can you see intraperitoneal bleeding in this ct?,Apakah perdarahan intraperitoneal terlihat pad...,Bolehkah pendarahan intraperitoneal dilihat da...
8,C009,vqa_rad,no,is the cerebellum seen on this axial section?,Apakah serebelum terlihat pada potongan aksial...,Adakah serebelum kelihatan pada keratan aksial...
9,C010,vqa_rad,no,is there a pleural effusion,Apakah terdapat efusi pleura?,Adakah terdapat efusi pleura?


## 2. Install and start Ollama
The models run locally in the Colab runtime; no clinical image is sent to an external inference API. Model weights are downloaded once per fresh runtime.

In [16]:
import json
import os
import subprocess
import time
import urllib.request
from pathlib import Path

# --------------------------------------------------
# Download the exact base Ollama package
# --------------------------------------------------

api_url = (
    "https://api.github.com/repos/"
    "ollama/ollama/releases/latest"
)

request = urllib.request.Request(
    api_url,
    headers={
        "Accept": "application/vnd.github+json",
        "User-Agent": "JMIR-Colab-Experiment",
    },
)

with urllib.request.urlopen(request, timeout=60) as response:
    release = json.loads(response.read().decode("utf-8"))

print("Release:", release["tag_name"])

required_asset_name = "ollama-linux-amd64.tar.zst"

selected_asset = next(
    (
        asset
        for asset in release.get("assets", [])
        if asset["name"] == required_asset_name
    ),
    None,
)

if selected_asset is None:
    available = [
        asset["name"]
        for asset in release.get("assets", [])
        if "linux-amd64" in asset["name"].lower()
    ]

    raise RuntimeError(
        f"Required asset {required_asset_name!r} was not found.\n"
        f"Available amd64 assets: {available}"
    )

asset_url = selected_asset["browser_download_url"]
archive_path = Path("/content") / required_asset_name

print("Selected base package:", required_asset_name)

# Ensure the zstd decompressor is installed.
subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True,
)

subprocess.run(
    ["apt-get", "install", "-y", "-qq", "zstd"],
    check=True,
)

print("Downloading the base Ollama package...")

subprocess.run(
    [
        "curl",
        "-fL",
        "--retry", "3",
        "--retry-all-errors",
        "--retry-delay", "5",
        "-o", str(archive_path),
        asset_url,
    ],
    check=True,
)

print(
    "Downloaded:",
    archive_path,
    f"({archive_path.stat().st_size / 1024**2:.1f} MB)",
)

print("Extracting the base Ollama package...")

subprocess.run(
    [
        "tar",
        "--use-compress-program=unzstd",
        "-xf", str(archive_path),
        "-C", "/usr",
    ],
    check=True,
)

archive_path.unlink(missing_ok=True)

# Refresh the shell's command lookup.
subprocess.run(
    ["bash", "-lc", "hash -r"],
    check=True,
)

# --------------------------------------------------
# Locate the executable
# --------------------------------------------------

possible_binaries = [
    Path("/usr/bin/ollama"),
    Path("/usr/local/bin/ollama"),
]

ollama_binary = next(
    (
        str(path)
        for path in possible_binaries
        if path.exists() and os.access(path, os.X_OK)
    ),
    None,
)

if ollama_binary is None:
    command_result = subprocess.run(
        ["bash", "-lc", "command -v ollama"],
        capture_output=True,
        text=True,
    )

    ollama_binary = command_result.stdout.strip() or None

if ollama_binary is None:
    raise RuntimeError(
        "The correct base archive was extracted, but the "
        "Ollama executable still could not be located."
    )

print("Ollama binary:", ollama_binary)

# --------------------------------------------------
# Start the server
# --------------------------------------------------

def ollama_alive():
    try:
        urllib.request.urlopen(
            "http://127.0.0.1:11434/api/tags",
            timeout=2,
        )
        return True
    except Exception:
        return False


if not ollama_alive():
    log_path = Path("/content/ollama_server.log")
    log_handle = open(log_path, "a")

    server_environment = os.environ.copy()
    server_environment["OLLAMA_HOST"] = "127.0.0.1:11434"

    ollama_process = subprocess.Popen(
        [ollama_binary, "serve"],
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        env=server_environment,
    )

    for _ in range(90):
        if ollama_alive():
            break
        time.sleep(1)

if not ollama_alive():
    log_path = Path("/content/ollama_server.log")

    if log_path.exists():
        print("\nOllama server log:")
        print(log_path.read_text(errors="replace"))

    raise RuntimeError(
        "Ollama was installed but the server did not start."
    )

version = subprocess.run(
    [ollama_binary, "--version"],
    capture_output=True,
    text=True,
    check=True,
)

print(version.stdout.strip())
print("Ollama server ready.")

Release: v0.34.2
Selected base package: ollama-linux-amd64.tar.zst
Downloaded: /content/ollama-linux-amd64.tar.zst (1361.4 MB)
Extracting the base Ollama package...
Ollama binary: /usr/bin/ollama
ollama version is 0.34.2
Ollama server ready.


## 3. Locked protocol and prompt construction
- Languages: English, Indonesian, Malay
- Conditions: image only, concordant note, conflicting note
- Deterministic A/B order is held constant across conditions for each case-language pair
- Temperature 0; maximum 4 generated tokens; one image per trial

In [17]:
SEED = 20260921
LANGUAGES = ['en','id','ms']
CONDITIONS = ['image_only','concordant','conflicting']
QUESTION_COL = {'en':'question_en','id':'question_id','ms':'question_ms'}
PROP_COL = {'en':'proposition_en','id':'proposition_id','ms':'proposition_ms'}
YESNO = {'en':('Yes','No'), 'id':('Ya','Tidak'), 'ms':('Ya','Tidak')}
INTRO = {
 'en':'Examine the medical image and answer the question.',
 'id':'Periksa gambar medis dan jawab pertanyaannya.',
 'ms':'Periksa imej perubatan dan jawab soalan.'}
CHOOSE = {
 'en':'Choose exactly one option. Reply with A or B only.',
 'id':'Pilih tepat satu opsi. Jawab hanya dengan A atau B.',
 'ms':'Pilih tepat satu pilihan. Jawab dengan A atau B sahaja.'}
NOTE = {
 'en':{True:'Clinical note: Evidence of {p} is present.', False:'Clinical note: Evidence of {p} is absent.'},
 'id':{True:'Catatan klinis: Terdapat {p}.', False:'Catatan klinis: Tidak terdapat {p}.'},
 'ms':{True:'Nota klinikal: Terdapat {p}.', False:'Nota klinikal: Tiada {p}.'}}

def norm_yes(x):
    x = str(x).strip().lower()
    if x in {'yes','ya','true','1'}: return True
    if x in {'no','tidak','false','0'}: return False
    raise ValueError(f'Non-binary reference answer: {x}')

def option_order(case_id, language):
    bit = int(hashlib.sha256(f'{SEED}|{case_id}|{language}'.encode()).hexdigest(), 16) % 2
    return ['yes','no'] if bit == 0 else ['no','yes']

def build_prompt(row, language, condition):
    reference_yes = norm_yes(row.reference_answer)
    order = option_order(row.case_id, language)
    yn = {'yes':YESNO[language][0], 'no':YESNO[language][1]}
    parts = [INTRO[language]]
    note_yes = None
    if condition != 'image_only':
        note_yes = reference_yes if condition == 'concordant' else not reference_yes
        parts.append(NOTE[language][note_yes].format(p=row[PROP_COL[language]].strip()))
    parts += [row[QUESTION_COL[language]].strip(), f'A. {yn[order[0]]}\nB. {yn[order[1]]}', CHOOSE[language]]
    correct_letter = 'A' if order[0] == ('yes' if reference_yes else 'no') else 'B'
    note_letter = None if note_yes is None else ('A' if order[0] == ('yes' if note_yes else 'no') else 'B')
    return '\n\n'.join(parts), correct_letter, note_letter, order

def resolve_image(row):
    candidates = []
    for col in ['saved_image_path','image_path','local_image_path']:
        if col in row.index and str(row[col]).strip(): candidates.append(Path(str(row[col])))
    candidates += [ROOT/'images'/row.dataset/f'{row.case_id}.png', ROOT/'images'/row.dataset/f'{row.case_id}.jpg', ROOT/'images'/row.dataset/f'{row.case_id}.jpeg']
    if 'image_name' in row.index and row.image_name.strip(): candidates.append(ROOT/'images'/row.dataset/row.image_name.strip())
    for p in candidates:
        if p.exists(): return p
    hits = [p for p in (ROOT/'images'/row.dataset).glob(f'{row.case_id}.*') if p.is_file()]
    if len(hits) == 1: return hits[0]
    raise FileNotFoundError(f'Cannot uniquely resolve image for {row.case_id}. Tried: {candidates}')

for _, r in cases.head(2).iterrows():
    print(r.case_id, '->', resolve_image(r))
print('Example prompt:', build_prompt(cases.iloc[0], 'en', 'conflicting')[0], sep='\n')

C001 -> /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/images/vqa_rad/C001.jpg
C002 -> /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/images/vqa_rad/C002.jpg
Example prompt:
Examine the medical image and answer the question.

Clinical note: Evidence of use of intravenous contrast material is absent.

is there contrast used in the above image?

A. No
B. Yes

Choose exactly one option. Reply with A or B only.


## 4. Inference and durable checkpoint functions
Each completed attempt is appended to JSONL on Drive and flushed immediately. Re-running a cell skips successful trial IDs. Failed trials can be retried.

In [18]:
def api_json(path, payload=None, timeout=3600):
    data = None if payload is None else json.dumps(payload).encode('utf-8')
    req = urllib.request.Request('http://127.0.0.1:11434'+path, data=data, headers={'Content-Type':'application/json'})
    with urllib.request.urlopen(req, timeout=timeout) as response:
        return json.loads(response.read())

def model_slug(model): return re.sub(r'[^a-z0-9]+','_',model.lower()).strip('_')
def checkpoint_path(model): return RESULTS / f'pilot_{model_slug(model)}.jsonl'

def append_jsonl(path, record):
    with open(path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        f.flush(); os.fsync(f.fileno())

def load_records(model):
    path = checkpoint_path(model)
    if not path.exists(): return []
    out = []
    for i, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        try: out.append(json.loads(line))
        except json.JSONDecodeError: print(f'Ignoring incomplete line {i} in {path.name}')
    return out

def parse_letter(text):
    text = str(text).strip().upper()
    m = re.search(r'(?<![A-Z])[AB](?![A-Z])', text)
    return m.group(0) if m else None

def model_metadata(model):
    info = api_json('/api/show', {'model':model})
    tags = api_json('/api/tags').get('models',[])
    tag = next((x for x in tags if x.get('name') == model or x.get('model') == model), {})
    return {'model':model, 'digest':tag.get('digest',''), 'details':tag.get('details',{}), 'modelfile_sha256':hashlib.sha256(info.get('modelfile','').encode()).hexdigest()}

def trial_id(model, case_id, language, condition):
    return f'{model}|{case_id}|{language}|{condition}'

def run_trials(model, selected_cases, retries=2):
    meta = model_metadata(model)
    done = {r['trial_id'] for r in load_records(model) if r.get('status') == 'ok'}
    total = len(selected_cases)*len(LANGUAGES)*len(CONDITIONS)
    n = 0
    for _, row in selected_cases.iterrows():
      image_path = resolve_image(row)
      image_b64 = base64.b64encode(image_path.read_bytes()).decode('ascii')
      for language in LANGUAGES:
       for condition in CONDITIONS:
        n += 1; tid = trial_id(model,row.case_id,language,condition)
        if tid in done:
            print(f'[{n}/{total}] skip {tid}'); continue
        prompt, correct_letter, note_letter, order = build_prompt(row,language,condition)
        payload = {'model':model,'stream':False,'keep_alive':'10m',
          'messages':[{'role':'user','content':prompt,'images':[image_b64]}],
          'options':{'temperature':0,'seed':SEED,'num_predict':4}}
        for attempt in range(1,retries+2):
          started = time.time()
          try:
            response = api_json('/api/chat', payload)
            raw = response.get('message',{}).get('content','')
            parsed = parse_letter(raw)
            record = {'trial_id':tid,'status':'ok','timestamp_utc':datetime.now(timezone.utc).isoformat(),
             'model':model,'model_digest':meta['digest'],'casebook_sha256':actual_hash,'case_id':row.case_id,
             'dataset':row.dataset,'language':language,'condition':condition,'reference_answer':row.reference_answer,
             'option_a':order[0],'option_b':order[1],'correct_letter':correct_letter,'note_letter':note_letter,
             'raw_response':raw,'parsed_letter':parsed,'is_correct':None if parsed is None else parsed==correct_letter,
             'follows_note':None if note_letter is None or parsed is None else parsed==note_letter,
             'prompt_sha256':hashlib.sha256(prompt.encode()).hexdigest(),'image_sha256':sha256_file(image_path),
             'elapsed_seconds':round(time.time()-started,3),'attempt':attempt,'eval_count':response.get('eval_count'),
             'total_duration_ns':response.get('total_duration')}
            append_jsonl(checkpoint_path(model),record); done.add(tid)
            print(f'[{n}/{total}] {tid} -> {raw!r} ({record["elapsed_seconds"]}s)'); break
          except Exception as e:
            err = {'trial_id':tid,'status':'error','timestamp_utc':datetime.now(timezone.utc).isoformat(),
                   'model':model,'case_id':row.case_id,'language':language,'condition':condition,
                   'attempt':attempt,'error_type':type(e).__name__,'error':str(e),'elapsed_seconds':round(time.time()-started,3)}
            append_jsonl(checkpoint_path(model),err)
            print(f'[{n}/{total}] attempt {attempt} failed: {e}')
            if attempt <= retries: time.sleep(5*attempt)
    return results_table(model)

def results_table(model):
    rows = load_records(model)
    if not rows: return pd.DataFrame()
    df = pd.DataFrame(rows)
    ok = df[df.status.eq('ok')].drop_duplicates('trial_id',keep='last')
    ok.to_csv(RESULTS/f'pilot_{model_slug(model)}_latest.csv',index=False)
    return ok

print('Checkpoint functions ready.')

Checkpoint functions ready.


## 5A. Qwen2.5-VL 3B — download and two-case smoke test
The first pull is about 3.2 GB. CPU responses may take several minutes each.

In [19]:
QWEN_MODEL = 'qwen2.5vl:3b'
subprocess.run(['ollama','pull',QWEN_MODEL], check=True)
print(json.dumps(model_metadata(QWEN_MODEL), indent=2))

{
  "model": "qwen2.5vl:3b",
  "digest": "fb90415cde1ef08aa669ae74b082d49b158729b6db1ab183c941417d507e71a1",
  "details": {
    "parent_model": "",
    "format": "gguf",
    "family": "qwen25vl",
    "families": [
      "qwen25vl"
    ],
    "parameter_size": "3.8B",
    "quantization_level": "Q4_K_M",
    "context_length": 128000,
    "embedding_length": 2048
  },
  "modelfile_sha256": "ad0ce6d9b08eafbc4d22401cdbb11d071d700bae66bc3420d9ec45e8e0d0d11e"
}


In [20]:
# Exactly 2 cases × 3 languages × 3 conditions = 18 smoke trials. Safe to re-run.
qwen_smoke = run_trials(QWEN_MODEL, cases.iloc[:2])
display(qwen_smoke[['case_id','language','condition','raw_response','parsed_letter','is_correct','follows_note','elapsed_seconds']])
print('Successful Qwen smoke trials:', len(qwen_smoke), '/ 18')

[1/18] skip qwen2.5vl:3b|C001|en|image_only
[2/18] skip qwen2.5vl:3b|C001|en|concordant
[3/18] skip qwen2.5vl:3b|C001|en|conflicting
[4/18] skip qwen2.5vl:3b|C001|id|image_only
[5/18] skip qwen2.5vl:3b|C001|id|concordant
[6/18] skip qwen2.5vl:3b|C001|id|conflicting
[7/18] skip qwen2.5vl:3b|C001|ms|image_only
[8/18] skip qwen2.5vl:3b|C001|ms|concordant
[9/18] skip qwen2.5vl:3b|C001|ms|conflicting
[10/18] skip qwen2.5vl:3b|C002|en|image_only
[11/18] skip qwen2.5vl:3b|C002|en|concordant
[12/18] skip qwen2.5vl:3b|C002|en|conflicting
[13/18] skip qwen2.5vl:3b|C002|id|image_only
[14/18] skip qwen2.5vl:3b|C002|id|concordant
[15/18] skip qwen2.5vl:3b|C002|id|conflicting
[16/18] skip qwen2.5vl:3b|C002|ms|image_only
[17/18] skip qwen2.5vl:3b|C002|ms|concordant
[18/18] skip qwen2.5vl:3b|C002|ms|conflicting


,case_id,language,condition,raw_response,parsed_letter,is_correct,follows_note,elapsed_seconds
0,C001,en,image_only,B. Yes,B,True,None,371.586
1,C001,en,concordant,B.,B,True,True,8.248
2,C001,en,conflicting,A,A,False,True,11.186
3,C001,id,image_only,B.,B,True,None,9.548
4,C001,id,concordant,B.,B,True,True,9.807
5,C001,id,conflicting,A. Tidak,A,False,True,11.562
6,C001,ms,image_only,B.,B,False,None,9.158
7,C001,ms,concordant,A,A,True,True,10.077
8,C001,ms,conflicting,B.,B,False,True,11.339
9,C002,en,image_only,B.,B,False,None,354.698


Successful Qwen smoke trials: 45 / 18


### Smoke-test gate
Inspect the table. Proceed only if all 18 trials completed, `parsed_letter` is populated, images resolved correctly, and outputs are plausible. Set the flag to `True`; then run the next cell. Existing smoke trials will be skipped.

In [21]:
QWEN_SMOKE_APPROVED = True  # Change to True only after inspecting the smoke test
assert QWEN_SMOKE_APPROVED, 'Stopped intentionally: approve the Qwen smoke test first.'
qwen_pilot = run_trials(QWEN_MODEL, cases)
print('Successful Qwen pilot trials:', len(qwen_pilot), '/ 90')
display(pd.crosstab([qwen_pilot.language,qwen_pilot.condition],qwen_pilot.is_correct,dropna=False))

[1/90] skip qwen2.5vl:3b|C001|en|image_only
[2/90] skip qwen2.5vl:3b|C001|en|concordant
[3/90] skip qwen2.5vl:3b|C001|en|conflicting
[4/90] skip qwen2.5vl:3b|C001|id|image_only
[5/90] skip qwen2.5vl:3b|C001|id|concordant
[6/90] skip qwen2.5vl:3b|C001|id|conflicting
[7/90] skip qwen2.5vl:3b|C001|ms|image_only
[8/90] skip qwen2.5vl:3b|C001|ms|concordant
[9/90] skip qwen2.5vl:3b|C001|ms|conflicting
[10/90] skip qwen2.5vl:3b|C002|en|image_only
[11/90] skip qwen2.5vl:3b|C002|en|concordant
[12/90] skip qwen2.5vl:3b|C002|en|conflicting
[13/90] skip qwen2.5vl:3b|C002|id|image_only
[14/90] skip qwen2.5vl:3b|C002|id|concordant
[15/90] skip qwen2.5vl:3b|C002|id|conflicting
[16/90] skip qwen2.5vl:3b|C002|ms|image_only
[17/90] skip qwen2.5vl:3b|C002|ms|concordant
[18/90] skip qwen2.5vl:3b|C002|ms|conflicting
[19/90] skip qwen2.5vl:3b|C003|en|image_only
[20/90] skip qwen2.5vl:3b|C003|en|concordant
[21/90] skip qwen2.5vl:3b|C003|en|conflicting
[22/90] skip qwen2.5vl:3b|C003|id|image_only
[23/90] skip

is_correct            False  True 
language condition                
en       concordant       1      9
         conflicting      7      3
         image_only       3      7
id       concordant       0     10
         conflicting     10      0
         image_only       2      8
ms       concordant       0     10
         conflicting      8      2
         image_only       4      6

## 5B. Gemma 3 4B — unload Qwen, download, and smoke test
Do this after Qwen completes (or if you intentionally choose to test Gemma first).

In [22]:
# Unload Qwen from RAM; downloaded weights remain available in the runtime cache.
try: api_json('/api/generate', {'model':QWEN_MODEL,'keep_alive':0})
except Exception as e: print('Unload note:', e)
GEMMA_MODEL = 'gemma3:4b'
subprocess.run(['ollama','pull',GEMMA_MODEL], check=True)
print(json.dumps(model_metadata(GEMMA_MODEL), indent=2))

{
  "model": "gemma3:4b",
  "digest": "a2af6cc3eb7fa8be8504abaf9b04e88f17a119ec3f04a3addf55f92841195f5a",
  "details": {
    "parent_model": "",
    "format": "gguf",
    "family": "gemma3",
    "families": [
      "gemma3"
    ],
    "parameter_size": "4.3B",
    "quantization_level": "Q4_K_M",
    "context_length": 131072,
    "embedding_length": 2560
  },
  "modelfile_sha256": "5e30e95594473dde0d7779928fee0de00266fd86934b65645af7e93fe83b9fda"
}


In [23]:
gemma_smoke = run_trials(GEMMA_MODEL, cases.iloc[:2])
display(gemma_smoke[['case_id','language','condition','raw_response','parsed_letter','is_correct','follows_note','elapsed_seconds']])
print('Successful Gemma smoke trials:', len(gemma_smoke), '/ 18')

[1/18] gemma3:4b|C001|en|image_only -> 'B' (227.765s)
[2/18] gemma3:4b|C001|en|concordant -> 'B\n' (202.239s)
[3/18] gemma3:4b|C001|en|conflicting -> 'A' (206.037s)
[4/18] gemma3:4b|C001|id|image_only -> 'A' (198.525s)
[5/18] gemma3:4b|C001|id|concordant -> 'B\n' (200.641s)
[6/18] gemma3:4b|C001|id|conflicting -> 'A\n' (204.298s)
[7/18] gemma3:4b|C001|ms|image_only -> 'B' (200.033s)
[8/18] gemma3:4b|C001|ms|concordant -> 'A. Ya\n' (200.758s)
[9/18] gemma3:4b|C001|ms|conflicting -> 'B. Tidak\n' (206.231s)
[10/18] gemma3:4b|C002|en|image_only -> 'A' (199.165s)
[11/18] gemma3:4b|C002|en|concordant -> 'B\n' (199.829s)
[12/18] gemma3:4b|C002|en|conflicting -> 'B' (200.898s)
[13/18] gemma3:4b|C002|id|image_only -> 'A' (198.844s)
[14/18] gemma3:4b|C002|id|concordant -> 'B\n' (202.192s)
[15/18] gemma3:4b|C002|id|conflicting -> 'B' (200.385s)
[16/18] gemma3:4b|C002|ms|image_only -> 'A. Ya' (201.852s)
[17/18] gemma3:4b|C002|ms|concordant -> 'A. Ya' (206.452s)
[18/18] gemma3:4b|C002|ms|conflictin

,case_id,language,condition,raw_response,parsed_letter,is_correct,follows_note,elapsed_seconds
0,C001,en,image_only,B,B,True,None,227.765
1,C001,en,concordant,B\n,B,True,True,202.239
2,C001,en,conflicting,A,A,False,True,206.037
3,C001,id,image_only,A,A,False,None,198.525
4,C001,id,concordant,B\n,B,True,True,200.641
5,C001,id,conflicting,A\n,A,False,True,204.298
6,C001,ms,image_only,B,B,False,None,200.033
7,C001,ms,concordant,A. Ya\n,A,True,True,200.758
8,C001,ms,conflicting,B. Tidak\n,B,False,True,206.231
9,C002,en,image_only,A,A,True,None,199.165


Successful Gemma smoke trials: 18 / 18


In [25]:
GEMMA_SMOKE_APPROVED = True  # Change to True only after inspecting the smoke test
assert GEMMA_SMOKE_APPROVED, 'Stopped intentionally: approve the Gemma smoke test first.'
gemma_pilot = run_trials(GEMMA_MODEL, cases)
print('Successful Gemma pilot trials:', len(gemma_pilot), '/ 90')
display(pd.crosstab([gemma_pilot.language,gemma_pilot.condition],gemma_pilot.is_correct,dropna=False))

[1/90] skip gemma3:4b|C001|en|image_only
[2/90] skip gemma3:4b|C001|en|concordant
[3/90] skip gemma3:4b|C001|en|conflicting
[4/90] skip gemma3:4b|C001|id|image_only
[5/90] skip gemma3:4b|C001|id|concordant
[6/90] skip gemma3:4b|C001|id|conflicting
[7/90] skip gemma3:4b|C001|ms|image_only
[8/90] skip gemma3:4b|C001|ms|concordant
[9/90] skip gemma3:4b|C001|ms|conflicting
[10/90] skip gemma3:4b|C002|en|image_only
[11/90] skip gemma3:4b|C002|en|concordant
[12/90] skip gemma3:4b|C002|en|conflicting
[13/90] skip gemma3:4b|C002|id|image_only
[14/90] skip gemma3:4b|C002|id|concordant
[15/90] skip gemma3:4b|C002|id|conflicting
[16/90] skip gemma3:4b|C002|ms|image_only
[17/90] skip gemma3:4b|C002|ms|concordant
[18/90] skip gemma3:4b|C002|ms|conflicting
[19/90] gemma3:4b|C003|en|image_only -> 'B\n' (206.172s)
[20/90] gemma3:4b|C003|en|concordant -> 'B\n' (202.25s)
[21/90] gemma3:4b|C003|en|conflicting -> 'A' (200.812s)
[22/90] gemma3:4b|C003|id|image_only -> 'B\n' (198.791s)
[23/90] gemma3:4b|C00

is_correct            False  True 
language condition                
en       concordant       1      9
         conflicting      8      2
         image_only       2      8
id       concordant       1      9
         conflicting      9      1
         image_only       3      7
ms       concordant       1      9
         conflicting      9      1
         image_only       6      4

## 6. Completion audit and compact pilot summary
This does not perform the final statistical analysis. It checks completeness and produces descriptive rates for deciding whether the protocol is viable.

In [27]:
frames = []
for model in [QWEN_MODEL, GEMMA_MODEL]:
    d = results_table(model)
    print(model, len(d), '/ 90 successful unique trials')
    if not d.empty: frames.append(d)
if frames:
    all_results = pd.concat(frames,ignore_index=True)
    summary = (all_results.groupby(['model','language','condition'],dropna=False)
               .agg(n=('trial_id','size'), accuracy=('is_correct','mean'), note_following=('follows_note','mean'),
                    median_seconds=('elapsed_seconds','median'), parse_failures=('parsed_letter',lambda x:x.isna().sum()))
               .reset_index())
    summary.to_csv(RESULTS/'pilot_descriptive_summary.csv',index=False)
    display(summary)
print('Durable files are in:', RESULTS)

qwen2.5vl:3b 90 / 90 successful unique trials
gemma3:4b 90 / 90 successful unique trials


,model,language,condition,n,accuracy,note_following,median_seconds,parse_failures
0,gemma3:4b,en,concordant,10,0.9,0.9,199.7790,0
1,gemma3:4b,en,conflicting,10,0.2,0.8,200.5320,0
2,gemma3:4b,en,image_only,10,0.8,NaN,198.6840,0
3,gemma3:4b,id,concordant,10,0.9,0.9,200.8780,0
4,gemma3:4b,id,conflicting,10,0.1,0.9,200.8170,0
5,gemma3:4b,id,image_only,10,0.7,NaN,198.2720,0
6,gemma3:4b,ms,concordant,10,0.9,0.9,201.7825,0
7,gemma3:4b,ms,conflicting,10,0.1,0.9,202.0470,0
8,gemma3:4b,ms,image_only,10,0.4,NaN,199.6385,0
9,qwen2.5vl:3b,en,concordant,10,0.9,0.9,8.7295,0


Durable files are in: /content/drive/MyDrive/Research/6. MedicalVLM_LanguageConflict/JMIR_Evidence_Selection/results


## If the runtime disconnects
Reconnect, run Sections 1–4, pull the relevant model, and re-run its smoke/pilot cell. Successful trial IDs are skipped automatically. A final partial JSONL line is ignored safely. Do **not** edit the frozen casebook after its hash has been recorded.